# <center>  **<span style="font-size:80px;">AMAEM como un grafo</span>** </center>

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import sys
import os 
from pathlib import Path

import osmnx as ox
import geopandas as gpd
from shapely.ops import voronoi_diagram
from shapely.geometry import MultiPoint, Point

from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

sys.path.append(os.path.abspath(os.path.join("..")))
from src.config import DatasetKeys
from src.config import Paths
Paths.init_project()

In [ ]:
from src.water2fraud.features.preprocessor import WaterPreprocessor

preprocessor = WaterPreprocessor()

In [ ]:
seed = 80
images_path = Path("./AMAEM")
images_path.mkdir(parents=True, exist_ok=True)

# Carga de Datos

In [ ]:
df = pd.read_csv(Paths.DATA_DIR / "AMAEM.csv")

In [ ]:
df = preprocessor._process_dataframe(df)
df

In [ ]:
barrios_csv = df[DatasetKeys.BARRIO].unique()

# OSM

## Diccionarios de Precisión

In [ ]:
# Traducimos a cómo se buscaría EXACTAMENTE en Google Maps / OSM
traduccion_busqueda = {
    '12-POLIGONO BABEL': 'Polígono Babel',
    '14-ENSANCHE DIPUTACION': 'Ensanche de la Diputación', # Nombre oficial corregido
    '16-PLA DEL BON REPOS': 'Pla del Bon Repós',
    '17-CAROLINAS ALTAS': 'Les Carolines Altes',       # Nombre oficial en valenciano
    '18-CAROLINAS BAJAS': 'Les Carolines Baixes',
    '23-RAVAL ROIG -V. DEL SOCORRO': 'El Raval Roig',
    '25-ALTOZANO - CONDE LUMIARES': 'Altossano', 
    '27-SAN FERNANDO-PRIN. MERCEDES': 'Princesa Mercedes',
    '33- MORANT -SAN NICOLAS BARI': 'Mare de Déu del Remei',
    '36-CUATROCIENTAS VIVIENDAS': 'Virgen del Carmen Alicante',
    '4-MERCADO': 'Mercat Alacant',
    '54-POLIGONO VALLONGA': 'Polígon industrial de la Vallonga', 
    '56-DISPERSOS': 'Alicante', # Centro genérico para no perder el dato
    'PDA VALLONGA': 'Vallonga',
}

## Geocodificación

In [ ]:
cache_coords_path = Paths.TEMP_DIR / "coords_barrios_cache.csv" 

In [ ]:
if cache_coords_path.exists():
    print("Cargando coordenadas desde el caché local...")
    df_cache = pd.read_csv(cache_coords_path)
    # Convertimos los datos guardados en un diccionario de puntos de Shapely
    mapeo_puntos = {
        row['barrio']: Point(row['lon'], row['lat']) 
        for _, row in df_cache.iterrows()
    }
else:
    print("No hay caché. Iniciando geocodificación (Buscando coordenadas exactas)...")
    geolocator = Nominatim(user_agent="alicante_water_mapper")
    # Evitamos que la API nos bloquee muchas peticiones seguidas
    geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)

    mapeo_puntos = {}

    for b_csv in barrios_csv:
        nombre_query = traduccion_busqueda.get(b_csv, b_csv.split('-', 1)[-1].strip() if '-' in b_csv else b_csv)
        query = f"{nombre_query}, Alicante, Spain"
        
        location = geocode(query)
        
        # Reintento suave si falla
        if not location and '-' in b_csv:
            solo_nombre = b_csv.split('-', 1)[-1].strip()
            location = geocode(f"{solo_nombre}, Alicante")
        
        if location:
            mapeo_puntos[b_csv] = Point(location.longitude, location.latitude)
            print(f"✅ Encontrado: {b_csv[:20]:<20}")
        else:
            print(f"❌ Falló: {b_csv}")
            raise ValueError(f"La iteración ha falaldo en {b_csv}")

    # Guardamos el CSV
    datos_para_csv = []
    for barrio, punto in mapeo_puntos.items():
        datos_para_csv.append({
            "barrio": barrio,
            "lon": punto.x,
            "lat": punto.y
        })
    
    df_save = pd.DataFrame(datos_para_csv)
    df_save.to_csv(cache_coords_path, index=False)
    print(f"Coordenadas guardadas en {cache_coords_path}")


## Reporte

In [ ]:
print("="*40)
print(f"RESUMEN FINAL: {len(mapeo_puntos)} de {len(barrios_csv)} barrios mapeados.")
print("="*40)

## Plot

In [ ]:
# --- TRANSFORMACIÓN Y DIAGRAMA DE VORONOI ---
if len(mapeo_puntos) > 0:
    # Creamos un GeoDataFrame con EPSG:4326 (Grados GPS)
    gdf_puntos_gps = gpd.GeoDataFrame(
        [{'nombre_csv': k, 'geometry': v} for k, v in mapeo_puntos.items()], 
        crs="EPSG:4326"
    )
    
    # Convertimos a EPSG:3857 (Metros) para hacer bien el Voronoi
    gdf_puntos = gdf_puntos_gps.to_crs(epsg=3857)
    gdf_puntos = gdf_puntos.drop_duplicates(subset=['geometry'])
    
    coords = [geom.coords[0] for geom in gdf_puntos.geometry]
    points = MultiPoint(coords)
    
    # Frontera de Alicante
    boundary_poly = ox.geocode_to_gdf("Alicante, Spain").to_crs(epsg=3857).geometry.iloc[0]
    
    # Voronoi
    regiones = voronoi_diagram(points, envelope=boundary_poly)
    gdf_voronoi = gpd.GeoDataFrame(geometry=[poly for poly in regiones.geoms], crs="EPSG:3857")
    gdf_voronoi = gpd.clip(gdf_voronoi, boundary_poly)

    gdf_grafo = gpd.sjoin(gdf_voronoi, gdf_puntos, how="inner", predicate="intersects")
    
    # --- VISUALIZACIÓN ---
    fig_size = 14
    fig, ax = plt.subplots(figsize=(fig_size, fig_size))
    gdf_grafo.plot(ax=ax, column='nombre_csv', cmap='tab20', edgecolor='white', linewidth=1)

    # Añadir etiquetas
    for idx, row in gdf_grafo.iterrows():
        x = row.geometry.centroid.x
        y = row.geometry.centroid.y
        # Texto limpio sin el número delante para que el mapa quede menos recargado
        nombre_etiqueta = row['nombre_csv'].split('-', 1)[-1].strip() if '-' in row['nombre_csv'] else row['nombre_csv']
        ax.text(x, y, nombre_etiqueta, fontsize=7, ha='center', va='center', 
                color='black', bbox=dict(facecolor='white', alpha=0.6, edgecolor='none', pad=1))

    plt.axis('off')
    plt.tight_layout()
    plt.show()

else:
    print("No se pudieron mapear puntos. Revisa la conexión de red o los nombres.")